# Evaluation of 3 Retrieval Approaches
1. Vector search using cosine similarity
2. Hybrid search: BM25 + vector search using cosine similarity
3. Hybrid search + reranking: BM25 + vector search using cosine similarity + reranking

In [10]:
import json
import pandas as pd
from IPython.display import display

from support_ops.helpers.config import ROOT, load_config
from support_ops.policy.rag.embeddings import read_embeddings
from support_ops.policy.rag.retriever import APPROACHES, RagSettings, shared_retriever

settings = RagSettings.from_config(load_config()["policy"]["rag"])
manifest, documents = read_embeddings()
cases = json.loads((ROOT / "data/cases/rag/questions.json").read_text())
ids = {doc["_id"] for doc in documents}
assert cases and len({case["id"] for case in cases}) == len(cases)
for case in cases:
    assert case["expected_chunk_ids"] and not set(case["expected_chunk_ids"]) - ids, case["id"]

# Evaluation metrics
- Recall@4: Fraction of expected chunks found in the final top 4 results

- MRR@4: How close to the absolute top slot the correct document is ranked (heavily rewards getting the answer to the #1 spot).

- nDCG@4: The overall quality of the rank order, penalising the system if highly relevant documents are placed at the bottom (ranks 3 or 4) instead of the top

Each expected chunk is labelled relevant (1). Other chunks are unlabelled (0).
Reference fetches are listed separately and excluded from these scores.

In [11]:
from ranx import evaluate

# Compare expected chunk IDs with the returned order; ranx calculates the metrics.
def retrieval_metrics(expected_ids, retrieved_ids, k):
    expected = {"query": {key: 1 for key in expected_ids}}
    # ranx needs scores as input; these numbers preserve the order already returned by retrieval.
    retrieved = {"query": {key: float(k - rank) for rank, key in enumerate(retrieved_ids[:k])}}
    scores = evaluate(expected, retrieved, [f"recall@{k}", f"mrr@{k}", f"ndcg@{k}"])
    return {name.replace(f"@{k}", "_at_k"): float(value) for name, value in scores.items()}

In [12]:
# Only evaluation selects different approaches; agent tools use retrieve(question).
retrievers = {
    name: shared_retriever(RagSettings.from_config(
        load_config(f"configs/rag/{name}.yaml")["policy"]["rag"]
    ))
    for name in APPROACHES
}

# Run every question through every approach with the same settings.
rows = []
for case in cases:
    for name, retriever in retrievers.items():
        hits = retriever.search(case["question"])
        retrieved = [hit["id"] for hit in hits if hit["via"] == "search"]
        references = [hit["id"] for hit in hits if hit["via"] != "search"]
        rows.append({
            "case_id": case["id"], "question": case["question"], "approach": name,
            "expected_chunk_ids": case["expected_chunk_ids"],
            "retrieved_chunk_ids": retrieved, "reference_chunk_ids": references,
            **retrieval_metrics(case["expected_chunk_ids"], retrieved, settings.k),
        })

# Show the average scores for each approach.
results = pd.DataFrame(rows)
metrics = ["recall_at_k", "mrr_at_k", "ndcg_at_k"]
summary = results.groupby("approach", sort=False)[metrics].mean().round(3)
summary.insert(0, "questions", len(cases))

display(summary)

,questions,recall_at_k,mrr_at_k,ndcg_at_k
approach,,,,
vector,24,0.938,0.875,0.880
hybrid,24,0.979,0.865,0.879
hybrid_rerank,24,0.979,0.910,0.922


In [13]:
# Save per-question results and the three-row summary as separate JSON files.
output = ROOT / "results/rag"
output.mkdir(parents=True, exist_ok=True)
(output / "retrieval_results.json").write_text(json.dumps({
    "k": settings.k, "candidate_pool": settings.n_prime, "rrf_k": settings.rrf_k,
    "corpus_hash": manifest["corpus_hash"], "rows": rows,
}, indent=2) + "\n")
(output / "retrieval_summary.json").write_text(
    json.dumps(summary.reset_index().to_dict(orient="records"), indent=2) + "\n"
)

392

In [14]:
# Inspect the expected, retrieved and referenced chunk IDs for one question.
CASE_ID = "q05"
case = next(case for case in cases if case["id"] == CASE_ID)
print(case["question"])
with pd.option_context("display.max_colwidth", None):
    display(results.loc[results.case_id == CASE_ID, [
        "approach", "expected_chunk_ids", "retrieved_chunk_ids", "reference_chunk_ids", "recall_at_k"
    ]].reset_index(drop=True))

The battery is leaking. I cannot safely switch it on to record evidence. Will my claim be rejected?


,approach,expected_chunk_ids,retrieved_chunk_ids,reference_chunk_ids,recall_at_k
0,vector,[returns-and-refunds-policy-3.2],"[returns-and-refunds-policy-7.5, returns-and-refunds-policy-3.2, returns-and-refunds-policy-4.6, orders-and-shipping-policy-7.4]","[returns-and-refunds-policy-3.1, orders-and-shipping-policy-7.1]",1.0
1,hybrid,[returns-and-refunds-policy-3.2],"[orders-and-shipping-policy-7.4, returns-and-refunds-policy-3.2, returns-and-refunds-policy-4.6, returns-and-refunds-policy-3.1]",[orders-and-shipping-policy-7.1],1.0
2,hybrid_rerank,[returns-and-refunds-policy-3.2],"[returns-and-refunds-policy-3.2, returns-and-refunds-policy-4.6, returns-and-refunds-policy-7.2, returns-and-refunds-policy-4.4]","[returns-and-refunds-policy-3.1, returns-and-refunds-policy-2.5]",1.0
